# VoiceArm — run the full pipeline on a free GPU**Code-switched speech grounding for open-vocabulary robotic manipulation.**Speak or type in Tamil, English, or Tanglish. A dual-ASR router transcribes it, alocal Qwen3-4B turns it into a JSON task plan, OWLv2 finds the named object in thecamera image, and an IK controller executes the pick-and-place in MuJoCo.Source: [GitHub](https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo)**Before you run:** turn on the GPU.* Kaggle — Settings, Accelerator, `GPU T4 x2`, and switch **Internet** on  (needs a phone-verified account, or the model downloads will fail)* Colab — Runtime, Change runtime type, `T4 GPU`Simulation only. No sim-to-real transfer is claimed.

## 1. System librariesMuJoCo renders offscreen, so it needs an EGL driver. This is the only apt step.

In [ ]:
import subprocess, sys, ossubprocess.run("apt-get -qq update", shell=True)subprocess.run("apt-get -qq install -y libegl1 libgles2 libosmesa6 libglfw3 > /dev/null",               shell=True)print("egl libraries installed")

## 2. Project and dependenciesThe Panda model is fetched by `setup.sh`, which pulls only the one robot directory (36 MB) instead of cloning all of `mujoco_menagerie`.

In [ ]:
import os, subprocess, sysfrom pathlib import PathROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")PROJ = ROOT / "voicearm"if not PROJ.exists():    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo.git", str(PROJ)], check=True)os.chdir(PROJ)sys.path.insert(0, str(PROJ))print("project at", PROJ)

In [ ]:
# The repo's requirements.txt pins the Apple/MLX stack; on an NVIDIA box we want# the portable one that the Space uses.!pip install -q -r hf_space/requirements.txt 2>&1 | tail -3!pip install -q sounddevice 2>&1 | tail -1print("deps installed")

In [ ]:
# Only the Franka Panda, not the whole menagerie.import subprocess, json, urllib.requestfrom pathlib import PathDIR = Path("assets/mujoco_menagerie/franka_emika_panda")BASE = "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/main/franka_emika_panda"(DIR / "assets").mkdir(parents=True, exist_ok=True)for f in ["panda.xml", "hand.xml", "scene.xml", "LICENSE"]:    urllib.request.urlretrieve(f"{BASE}/{f}", DIR / f)api = "https://api.github.com/repos/google-deepmind/mujoco_menagerie/contents/franka_emika_panda/assets"meshes = json.load(urllib.request.urlopen(api))for m in meshes:    if m["type"] == "file":        urllib.request.urlretrieve(m["download_url"], DIR / "assets" / m["name"])print(f"{len(list((DIR/'assets').iterdir()))} mesh assets")

## 3. Backend check`src/backend.py` picks the model stack at import. MLX does not exist here, so it selects the transformers path and CUDA. `MUJOCO_GL` is set in `src/__init__.py` before anything imports mujoco.

In [ ]:
os.environ["VOICEARM_BACKEND"] = "torch"from src import backendprint(backend.describe())import torchprint("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")print("vram:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB"      if torch.cuda.is_available() else "-")

## 4. Simulation and kinematicsNo models needed yet — this is physics, rendering and IK only.

In [ ]:
!python scripts/m1_move.py!python scripts/m2_ik.py

In [ ]:
from PIL import ImageImage.open("out/m1_frames.png")

## 5. Pick and placeGrasps each block from its ground-truth pose. Perception comes next.

In [ ]:
!python scripts/m3_pick.py

In [ ]:
from IPython.display import VideoVideo("out/m3_pickplace.mp4", embed=True, width=640)

## 6. Open-vocabulary perceptionOWLv2 locates each object from a free-text description, then the box is unprojected with the depth buffer. Downloads ~1.5 GB.

In [ ]:
!python scripts/m4_detect.py

In [ ]:
from PIL import ImageImage.open("out/m4_detections.png")

## 7. LanguageDownloads Qwen3-4B (~8 GB). Runs eight utterances — English, Tanglish, and Tamil script — end to end through plan, perception and motion.

In [ ]:
!python scripts/m5_plan.py

## 8. ASR benchmarkCharacter error rate for both ASR backends on Tamil. Sentences are synthesised, so these are a floor on error, not real-speech accuracy. Needs a Tamil TTS voice, which Linux images do not ship — so this cell reads the wavs committed to the repo instead when `say` is missing.

In [ ]:
!python scripts/bench_asr.py || echo "no Tamil system voice here -- see the README table"

## 9. Live demoLaunches the Gradio app with a public share link, valid while this sessionlives. Record yourself in Tamil, English or Tanglish, or type an instruction.The share URL dies when the session ends, which is why the permanent link is astatic page rather than this.

In [ ]:
import importlib.util, sys
sys.path.insert(0, '.')

spec = importlib.util.spec_from_file_location('space_app', 'hf_space/app.py')
space_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(space_app)

space_app.warm_models()
space_app.build().queue(max_size=8).launch(share=True)

## 10. Optional: does a Tamil-native 24B planner do better?`sarvam-m` is the strongest open model for Tamil, and the local project rejectedit for one reason only — at 4-bit it needs about 13 GB and would not co-residewith the other three models on a 16 GB laptop. A T4 x2 session has 32 GB, so thecomparison the README could not make is possible here.This is expensive to download (~48 GB in fp16, less quantized). Skip it unlessyou specifically want that number.

In [ ]:
RUN_ABLATION = False   # set True to download sarvam-m and compareif RUN_ABLATION:    !pip install -q bitsandbytes accelerate    !python scripts/bench_planner.py --models Qwen/Qwen3-4B-Instruct-2507 sarvamai/sarvam-m --load-4bitelse:    print("skipped -- set RUN_ABLATION = True")